In [ ]:
from pwn import *
from Crypto.Protocol.KDF import SP800_108_Counter
from primitives import CMAC_PRF, SHA256
from cmac_solver import find_custom_context

In [227]:
HOST = 'fc8759e1de715f2403cdf8f7-9999-kdf-dream.challenge.cscg.live'
P = 0xFFFFFFFFFFFFFFFFC90FDAA22168C234C4C6628B80DC1CD129024E088A67CC74020BBEA63B139B22514A08798E3404DDEF9519B3CD3A431B302B0A6DF25F14374FE1356D6D51C245E485B576625E7EC6F44C42E9A637ED6B0BFF5CB6F406B7EDEE386BFB5A899FA5AE9F24117C4B1FE649286651ECE45B3DC2007CB8A163BF0598DA48361C55D39A69163FA8FD24CF5F83655D23DCA3AD961C62F356208552BB9ED529077096966D670C354E4ABC9804F1746C08CA18217C32905E462E36CE3BE39E772C180E86039B2783A2EC07A28FB5C55DF06F4C52C9DE2BCBF6955817183995497CEA956AE515D2261898FA051015728E5A8AACAA68FFFFFFFFFFFFFFFF
G = 2

In [228]:
conn = remote(HOST, 1337, ssl=True)

[x] Opening connection to fc8759e1de715f2403cdf8f7-9999-kdf-dream.challenge.cscg.live on port 1337
[x] Opening connection to fc8759e1de715f2403cdf8f7-9999-kdf-dream.challenge.cscg.live on port 1337: Trying 147.75.204.231
[+] Opening connection to fc8759e1de715f2403cdf8f7-9999-kdf-dream.challenge.cscg.live on port 1337: Done


In [229]:
def xor(a, b):
    return bytes([x ^ y for x, y in zip(a, b)])

def mitm_recv(new_msg=None):
    _from = conn.recvline(keepends=False).decode().split(' ')[3]
    ctx = conn.recvline(keepends=False).decode().split(': ')[-1]
    msg = conn.recvline(keepends=False).decode().split(': ')[-1]

    if new_msg is not None:
        mitm_send(new_msg)

    return msg, ctx, _from

def mitm_send(msg):
    conn.sendline(msg.encode())

In [230]:
my_x = 2
my_public = pow(G,my_x, P)
bob_public, _, _ = mitm_recv(str(my_public))
alice_public, _, _ = mitm_recv(str(my_public))
bob_public, alice_public

('9410568011161321266784803002125315657032613793309535376230207944448440507683048631123189543682045960538266591800158549132595983418339273680792165157975417364583838348712751061966808749622587829514842799924801068644957673559455228210259300738615599392459642731604087779314239128939862946331008509560858428885511716109748409035374517670096286871615328694447293085953339603869074303116360748720777800894663027928835677604801581414491786993430366632104473441648664560404859184268154706718578696712438046033864475159860922364922568943961011979714414710129725090257869134975254407189413253167208070546671110526791186077047',
 '17405439376548513030635897824978223859267306053670818968993311285406096233795512461750033503890842360882733592350847709565350335299787060986150723605888490809310231029078615763120293098530234619210591208427086573240804745047436954378210394966127525428766271200404090553731761531378187131830096130145701968929356528552718721684421677845942829179151416735465591970929772856303985

In [231]:
alice_shared_key = SHA256.new(str(pow(int(alice_public), my_x, P)).encode()).digest()[:16]
bob_shared_key = SHA256.new(str(pow(int(bob_public), my_x, P)).encode()).digest()[:16]

In [232]:
mitm_recv('CMAC')
mitm_recv('CMAC')

('HMAC/CMAC/KMAC', 'protocol', 'like')

In [233]:
second_ctx_half, _, _ = mitm_recv()
mitm_send(second_ctx_half)
first_ctx_half, _, _ = mitm_recv()
# Omit sending first_ctx_half to Bob, so we can calculate a custom ctx
first_ctx_half, second_ctx_half

('b468926b152fc3d3c55024544c13f320', '9db885688f45ec155c281005b9c5b910')

In [234]:
alice_ctx = bytes.fromhex(first_ctx_half) + bytes.fromhex(second_ctx_half)
alice_OTP = SP800_108_Counter(alice_shared_key, 16, CMAC_PRF, 1, b'keygen_for_secure_bagdrop', alice_ctx)
alice_msg = b'wearecompromised'
bagdrop = xor(alice_msg, alice_OTP)
alice_ctx, alice_OTP, alice_msg, bagdrop

(b'\xb4h\x92k\x15/\xc3\xd3\xc5P$TL\x13\xf3 \x9d\xb8\x85h\x8fE\xec\x15\\(\x10\x05\xb9\xc5\xb9\x10',
 b'\xda\x971\xbe\xdf\xf5F\xaa\xbe`\x9f\x96\x8e\x89wq',
 b'wearecompromised',
 b'\xad\xf2P\xcc\xba\x96)\xc7\xce\x12\xf0\xfb\xe7\xfa\x12\x15')

In [235]:
try:
    custom_first_half = find_custom_context(bob_shared_key, bytes.fromhex(second_ctx_half), bagdrop)
    print("Found custom ctx")
    print(custom_first_half.hex())
    mitm_send(custom_first_half.hex())
except:
    print("Error while generating ctx")
    conn.close()
    raise StopIteration()

[x] Bruteforcing
[x] Bruteforcing: Trying "07cf", 3.052%
[x] Bruteforcing: Trying "0f9f", 6.104%
[x] Bruteforcing: Trying "176f", 9.155%
[x] Bruteforcing: Trying "1f3f", 12.207%
[+] Bruteforcing: Found key: "2031"
SUCCESS
Found custom ctx
2031823719d5d5a15d4138fbf006055f


In [236]:
try:
    while True:
        print(conn.recvline().decode())
except:
    pass
finally:
    conn.close()

what would you like Bob to receive?Alice has made the bag drop, waiting for Bob to pick up and decode the message

dach2025{But_n1st_said_it_was_fine?!???_15f7a069}



[*] Closed connection to fc8759e1de715f2403cdf8f7-9999-kdf-dream.challenge.cscg.live port 1337
